# JEPA from Scratch: Predicting in Latent Space

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/jepa_world_models.ipynb)

A tiny **Joint-Embedding Predictive Architecture** (I-JEPA, Assran et al. 2023) built from
scratch in PyTorch on FashionMNIST. Instead of reconstructing pixels, the model predicts the
*latent representation* of a masked region from the surrounding context.

You will:
1. Build a tiny Vision-Transformer encoder and a predictor.
2. Train with no labels by predicting masked-region representations.
3. Linear-probe the frozen encoder and watch its latent space self-organise.
4. Reproduce **representation collapse** and see how the EMA target + stop-gradient cures it.
5. Compare latent prediction against a pixel-prediction baseline.

This notebook uses a small epoch count so it runs quickly. The blog post uses 35 epochs and a
larger masked region for its figures; raise `EPOCHS` for a stronger linear probe.


In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

## Patches

Each 28x28 image becomes a 4x4 grid of 7x7 patches (16 tokens), the tokens-from-an-image idea
behind Vision Transformers. Talking in patches lets us define "blocks" as sets of patch indices.

In [ ]:
GRID, PATCH = 4, 7              # 4x4 grid of 7x7 patches
N_PATCH = GRID * GRID           # 16 patches
PATCH_DIM = PATCH * PATCH       # 49 pixels per patch
DIM, PRED_DIM = 128, 64         # encoder / predictor widths
DEPTH, PRED_DEPTH, HEADS = 4, 2, 4

def patchify(x):
    """(B,1,28,28) -> (B,16,49) row-major grid of flattened 7x7 patches."""
    b = x.shape[0]
    p = x.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)
    return p.contiguous().view(b, N_PATCH, PATCH_DIM)

## The encoder and predictor

The **encoder** is a small Transformer over patch embeddings plus positional embeddings. The
**predictor** takes encoded context tokens plus learnable mask tokens (carrying only position)
and predicts the target representations.

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(PATCH_DIM, DIM)
        self.pos = nn.Parameter(torch.zeros(1, N_PATCH, DIM))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(DIM, HEADS, DIM * 2, dropout=0.0,
            activation="gelu", batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, DEPTH)
        self.norm = nn.LayerNorm(DIM)

    def forward(self, patches, idx):
        h = self.embed(patches) + self.pos[:, idx, :]
        return self.norm(self.blocks(h))


class Predictor(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.proj_in = nn.Linear(DIM, PRED_DIM)
        self.pos = nn.Parameter(torch.zeros(1, N_PATCH, PRED_DIM))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, PRED_DIM))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        layer = nn.TransformerEncoderLayer(PRED_DIM, HEADS, PRED_DIM * 2, dropout=0.0,
            activation="gelu", batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, PRED_DEPTH)
        self.head = nn.Linear(PRED_DIM, out_dim)

    def forward(self, ctx, ctx_idx, tgt_idx):
        b = ctx.shape[0]
        c = self.proj_in(ctx) + self.pos[:, ctx_idx, :]
        m = self.mask_token.expand(b, tgt_idx.shape[0], -1) + self.pos[:, tgt_idx, :]
        h = self.blocks(torch.cat([c, m], dim=1))
        return self.head(h[:, ctx.shape[1]:, :])

## Masking

We hide a contiguous region of 6 patches (the **target**) and keep the other 10 as **context**.
A large hidden region makes pixel reconstruction ambiguous, the regime where predicting an
abstraction starts to pay off.

In [ ]:
def sample_mask(rng):
    if rng.random() < 0.5:                       # 2 rows x 3 cols
        r, c = int(rng.integers(0, GRID - 1)), int(rng.integers(0, GRID - 2))
        tgt = [(r + i) * GRID + (c + j) for i in range(2) for j in range(3)]
    else:                                        # 3 rows x 2 cols
        r, c = int(rng.integers(0, GRID - 2)), int(rng.integers(0, GRID - 1))
        tgt = [(r + i) * GRID + (c + j) for i in range(3) for j in range(2)]
    ctx = [i for i in range(N_PATCH) if i not in set(tgt)]
    return np.array(ctx), np.array(tgt)

## Data and a probing helper

The probe representation of an image is the mean of the encoder's 16 patch tokens.

In [ ]:
tf = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.FashionMNIST("./data", train=True, download=True, transform=tf)
test_ds = datasets.FashionMNIST("./data", train=False, download=True, transform=tf)

N_TRAIN_PROBE, N_TEST = 5000, 1000
train_x = torch.stack([train_ds[i][0] for i in range(N_TRAIN_PROBE)])
train_y = np.array([train_ds[i][1] for i in range(N_TRAIN_PROBE)])
test_x = torch.stack([test_ds[i][0] for i in range(N_TEST)])
test_y = np.array([test_ds[i][1] for i in range(N_TEST)])
CLASSES = ["T-shirt","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Boot"]

@torch.no_grad()
def embed_images(encoder, x):
    encoder.eval()
    all_idx = torch.arange(N_PATCH, device=DEVICE)
    out = [encoder(patchify(x[i:i+512].to(DEVICE)), all_idx).mean(1).cpu().numpy()
           for i in range(0, x.shape[0], 512)]
    encoder.train()
    return np.concatenate(out)

## The training loop

One function trains all three variants:
- `jepa`: predict **latent** target codes; EMA target encoder with stop-gradient (the real recipe).
- `collapse`: targets from the **online** encoder with gradients flowing (no EMA, no stop-grad).
- `pixel`: predict the target region's **pixels** (a masked-autoencoder baseline).

In [ ]:
BATCH, LR, EMA_BASE = 256, 1e-3, 0.996
EPOCHS = 12   # raise to ~35 for a stronger probe (matches the blog figures)

def train_run(mode):
    torch.manual_seed(0); rng = np.random.default_rng(0)
    enc = Encoder().to(DEVICE)
    pred = Predictor(PATCH_DIM if mode == "pixel" else DIM).to(DEVICE)
    tgt_enc = copy.deepcopy(enc).to(DEVICE) if mode == "jepa" else None
    if tgt_enc:
        for p in tgt_enc.parameters():
            p.requires_grad_(False)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(pred.parameters()),
                            lr=LR, weight_decay=0.05)
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    total, step, stds = EPOCHS * len(loader), 0, []
    all_idx = torch.arange(N_PATCH, device=DEVICE)

    for epoch in range(EPOCHS):
        for xb, _ in loader:
            xb = xb.to(DEVICE); patches = patchify(xb)
            ci, ti = sample_mask(rng)
            ctx = enc(patches[:, ci, :], torch.as_tensor(ci, device=DEVICE))
            out = pred(ctx, torch.as_tensor(ci, device=DEVICE), torch.as_tensor(ti, device=DEVICE))
            if mode == "pixel":
                target = patches[:, ti, :]
            elif mode == "jepa":
                with torch.no_grad():
                    target = tgt_enc(patches, all_idx)[:, ti, :]
            else:
                target = enc(patches, all_idx)[:, ti, :]
            loss = F.mse_loss(out, target)
            opt.zero_grad(); loss.backward(); opt.step()
            if tgt_enc:
                tau = EMA_BASE + (1 - EMA_BASE) * (step / total)
                with torch.no_grad():
                    for pe, po in zip(tgt_enc.parameters(), enc.parameters()):
                        pe.mul_(tau).add_(po, alpha=1 - tau)
            step += 1
        stds.append(float(embed_images(enc, test_x).std(0).mean()))
        print(f"[{mode}] epoch {epoch:2d}  loss {loss.item():.4f}  emb_std {stds[-1]:.4f}")
    return enc, stds

## Train JEPA and watch the latent space

We train the healthy JEPA, then project the frozen encoder's test embeddings to 2D with PCA.
The colours are true labels, used only for the plot. The model never sees them.

In [ ]:
enc_jepa, std_jepa = train_run("jepa")

emb = embed_images(enc_jepa, test_x)
mu = emb.mean(0); _, _, vt = np.linalg.svd(emb - mu, full_matrices=False)
proj = (emb - mu) @ vt[:2].T
plt.figure(figsize=(6, 6))
for c in range(10):
    m = test_y == c
    plt.scatter(proj[m, 0], proj[m, 1], s=8, alpha=0.7, label=CLASSES[c])
plt.legend(fontsize=7, markerscale=1.5); plt.xticks([]); plt.yticks([])
plt.title("JEPA latent space (no labels used in training)"); plt.show()

## Linear probe

Freeze the encoder, fit logistic regression on its features, and compare to a random-init encoder.

In [ ]:
def probe(encoder):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(embed_images(encoder, train_x), train_y)
    return clf.score(embed_images(encoder, test_x), test_y)

torch.manual_seed(0); enc_rand = Encoder().to(DEVICE)
acc_rand, acc_jepa = probe(enc_rand), probe(enc_jepa)
print(f"random-init: {acc_rand:.3f}")
print(f"JEPA       : {acc_jepa:.3f}")

## Representation collapse

Remove the EMA target and the stop-gradient (the `collapse` variant). The loss falls to almost
zero, but the embeddings shrink to a single point: the model found the constant-vector cheat.

In [ ]:
enc_col, std_col = train_run("collapse")

plt.figure(figsize=(7, 4))
plt.plot(std_jepa, "-o", label="Healthy (EMA + stop-grad)")
plt.plot(std_col, "-o", color="crimson", label="Collapsed (no stop-grad)")
plt.xlabel("epoch"); plt.ylabel("mean embedding std"); plt.legend()
plt.title("The collapse signal"); plt.show()
print(f"healthy std {std_jepa[-1]:.3f}   collapsed std {std_col[-1]:.4f}")

## Latent prediction vs pixel prediction

Change only the target: predict the masked region's **pixels** instead of its latent code, then
probe. On smooth FashionMNIST the two are close (pixels can be a hair ahead). The latent
approach's advantage grows with how much unpredictable detail the images contain, which is why
I-JEPA's gains show at ImageNet scale.

In [ ]:
enc_pixel, _ = train_run("pixel")
acc_pixel = probe(enc_pixel)

plt.figure(figsize=(6, 4))
names, accs = ["Random", "Pixel-MAE", "JEPA"], [acc_rand, acc_pixel, acc_jepa]
plt.bar(names, [a * 100 for a in accs], color=["#9ca3af", "#f59e0b", "#2563eb"])
for i, a in enumerate(accs):
    plt.text(i, a * 100 + 1, f"{a*100:.1f}%", ha="center", fontweight="bold")
plt.ylabel("Linear-probe accuracy (%)"); plt.ylim(0, 100)
plt.title("Frozen-encoder quality on FashionMNIST"); plt.show()

## Exercises

1. **More epochs.** Raise `EPOCHS` to 35 and re-run. How much does the JEPA probe improve?
2. **Mask size.** Change `sample_mask` to hide a single 2x2 block. Does pixel prediction get
   relatively stronger when the masked region is small and easy?
3. **EMA momentum.** Set `EMA_BASE = 0.0` (target encoder tracks the online encoder instantly).
   Does the healthy run start to collapse?
4. **Predictor depth.** Set `PRED_DEPTH = 0` so the predictor is a single linear layer. How does
   that affect the learned representation?
5. **Harder data.** Swap FashionMNIST for CIFAR-10 (3 channels, richer texture). Does latent
   prediction now beat the pixel baseline?

## Where to go next

- Assran et al. (2023), *I-JEPA* (arXiv:2301.08243) and the official `facebookresearch/ijepa` code.
- LeCun (2022), *A Path Towards Autonomous Machine Intelligence*.
- V-JEPA and V-JEPA 2 for the video and planning extensions.
